# L6b Lab: Flux Balance in the HL-60 Urea-Cycle Network

This lab retains the compact Fall 2025 urea-cycle case and replaces the old student/solution split with one complete, validated notebook.

> **Learning objectives**
>
> - Parse a reaction file and assemble a stoichiometric matrix.
> - Explain why steady state is the linear constraint $S v=0$.
> - Encode reversibility and enzyme capacity as flux bounds.
> - Maximize urea export and independently check residuals and bounds.


## Setup and reaction data

Run the local setup cell first. It activates the pinned course environment, loads every package used by this meeting, and includes the `Week06Core` module from [`../src/Week06Core.jl`](../src/Week06Core.jl), which provides the functions called below. The reaction network parsed below is committed at [`data/Network.net`](data/Network.net).

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and includes the meeting's local source. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


The setup cell completed, so the environment and this lab's local source are loaded. The cell below reads the urea-cycle network file, storing its location in `reaction_path::String`, the parsed reactions in `reactions::Vector{Reaction}`, and the stoichiometry in `form::NamedTuple`, whose fields are the matrix `S::Matrix{Float64}`, the row labels `species::Vector{String}`, and the column labels `names::Vector{String}`.


In [2]:
reaction_path = joinpath(CHEME5800_L6B_DATA, "Network.net")
reactions = parse_reaction_file(reaction_path)
form = build_stoichiometric_matrix(reactions)
(size = size(form.S), reactions = form.names, species = form.species)


(size = (18, 19), reactions = ["v1", "v2", "v3", "v4", "v5", "b1", "b2", "b3", "b4", "b5", "b6", "b7", "b8", "b9", "b10", "b11", "b12", "b13", "b14"], species = ["M_L-Aspartate_c", "M_ATP_c", "M_L-Citrulline_c", "M_AMP_c", "M_N-(L-Arginino)succinate_c", "M_Diphosphate_c", "M_L-Arginine_c", "M_Fumarate_c", "M_H2O_c", "M_L-Ornithine_c", "M_Urea_c", "M_Carbamoyl_phosphate_c", "M_Orthophosphate_c", "M_H_c", "M_NADPH_c", "M_Oxygen_c", "M_NADP_c", "M_Nitric_oxide_c"])

## Formulation

For reaction flux vector $v$, the $i,j$ entry of $S$ is negative for a consumed species and positive for a produced species. The steady-state FBA problem is

$$\max_v -v_{b4}\quad\text{s.t.}\quad Sv=0,\qquad \ell\le v\le u,$$

where `b4` is written as an exchange *into* the system; export therefore has negative sign.


In [3]:
result = solve_urea_fba(reactions)
flux_table = DataFrame(reaction = result.names, flux = result.flux,
    lower = result.lower, upper = result.upper)
pretty_table(flux_table)
(maximum_urea_export = result.objective,
 maximum_balance_residual = norm(result.residual, Inf))


┌──────────┬─────────┬─────────┬─────────┐
│ reaction │    flux │   lower │   upper │
│   String │ Float64 │ Float64 │ Float64 │
├──────────┼─────────┼─────────┼─────────┤
│       v1 │  0.0328 │     0.0 │     0.1 │
│       v2 │  0.0328 │     0.0 │  0.0328 │
│       v3 │  0.0328 │     0.0 │     1.9 │
│       v4 │  0.0328 │     0.0 │     4.1 │
│       v5 │     0.0 │     0.0 │     0.1 │
│       b1 │  0.0328 │   -10.0 │    10.0 │
│       b2 │  0.0328 │   -10.0 │    10.0 │
│       b3 │ -0.0328 │   -10.0 │    10.0 │
│       b4 │ -0.0328 │   -10.0 │    10.0 │
│       b5 │  0.0328 │   -10.0 │    10.0 │
│       b6 │ -0.0328 │   -10.0 │    10.0 │
│       b7 │ -0.0328 │   -10.0 │    10.0 │
│       b8 │ -0.0328 │   -10.0 │    10.0 │
│       b9 │     0.0 │   -10.0 │    10.0 │
│      b10 │     0.0 │   -10.0 │    10.0 │
│      b11 │     0.0 │   -10.0 │    10.0 │
│      b12 │     0.0 │   -10.0 │    10.0 │
│      b13 │     0.0 │   -10.0 │    10.0 │
│      b14 │  0.0328 │   -10.0 │    10.0 │
└──────────

(maximum_urea_export = 0.03279999999999994, maximum_balance_residual = 6.245004513516506e-17)

In [4]:
@test size(result.S) == (18, 19)
@test result.objective ≈ 0.0328
@test norm(result.residual, Inf) < 1e-10
@test all(result.lower .- 1e-10 .<= result.flux .<= result.upper .+ 1e-10)
:fba_solution_verified


:fba_solution_verified

## Interpretation

The pathway flux is limited by reaction `v2` at 0.0328. The model does not prove that a cell realizes this state; it identifies the best steady-state flux permitted by the stated network and bounds. Biological interpretation must therefore cite both the objective and the assumptions.
